# Kaggle-Ready GPU Framework Validation

Upload this notebook to Kaggle and run it there.

Requirements:
- enable **GPU** accelerator
- enable **Internet**

This notebook will:
1. Clone the repo from GitHub
2. Apply the PLD compatibility fix if needed
3. Install the project and dependencies
4. Run the GPU-scale framework validation
5. Produce one bundled zip artifact so you do not need to download files one by one


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/mohdfaour03/EECE_608.git'
REPO_BRANCH = 'codex/kaggle-validation'
REPO_DIR = Path('/kaggle/working/EECE_608')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)
], check=True)

branch_checkout = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH],
    text=True,
    capture_output=True,
)
if branch_checkout.returncode == 0:
    subprocess.run([
        'git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'
    ], check=True)
else:
    print('branch fetch failed; falling back to default branch')
    print(branch_checkout.stderr)

head_commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'],
    text=True,
).strip()

print('python =', sys.executable)
print('repo_dir =', REPO_DIR)
print('repo_branch =', head_commit)
os.chdir(REPO_DIR)


In [ ]:
from pathlib import Path

pld_path = REPO_DIR / 'src' / 'dp_audit_tightness' / 'privacy' / 'pld_accounting.py'
text = pld_path.read_text(encoding='utf-8')

if 'from dp_accounting.pld import common' in text:
    text = text.replace('from dp_accounting.pld import common\n', '')

if '_VALUE_DISCRETIZATION_INTERVAL = 1e-4' not in text:
    text = text.replace(
        'import math\nfrom typing import Any\n',
        'import math\nfrom typing import Any\n\n_VALUE_DISCRETIZATION_INTERVAL = 1e-4\n',
    )

old_block = '''    pld_single = pld_lib.from_gaussian_mechanism(\n        standard_deviation=noise_multiplier,\n        sensitivities=[1.0],\n        pessimistic_estimate=True,\n        sampling_prob=sampling_rate,\n        use_connect_dots=True,\n        value_discretization_interval=common.VALUE_DISCRETIZATION_INTERVAL,\n    )\n'''

new_block = '''    kwargs = {\n        "standard_deviation": noise_multiplier,\n        "pessimistic_estimate": True,\n        "sampling_prob": sampling_rate,\n        "use_connect_dots": True,\n        "value_discretization_interval": _VALUE_DISCRETIZATION_INTERVAL,\n    }\n    try:\n        pld_single = pld_lib.from_gaussian_mechanism(\n            sensitivity=1.0,\n            **kwargs,\n        )\n    except TypeError:\n        pld_single = pld_lib.from_gaussian_mechanism(\n            sensitivities=[1.0],\n            **kwargs,\n        )\n'''

if old_block in text:
    text = text.replace(old_block, new_block)

pld_path.write_text(text, encoding='utf-8')
print('patched =', pld_path)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.4'], check=True)
print('install complete')


In [ ]:
import torch

print('torch_version =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
print('device_count =', torch.cuda.device_count())
print('arch_list =', torch.cuda.get_arch_list() if torch.cuda.is_available() else [])
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Enable GPU in the Kaggle notebook settings before running.')


In [ ]:
script_path = REPO_DIR / 'codex' / 'run_framework_validation_gpu_scale.py'
cmd = [sys.executable, str(script_path)]
print('running =', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
results_dir = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale'
expected = [
    results_dir / 'framework_validation_gpu_scale_summary.json',
    results_dir / 'framework_validation_gpu_scale_rows.csv',
    results_dir / 'framework_validation_gpu_scale_checks.json',
    results_dir / 'framework_validation_gpu_scale_report.md',
    results_dir / 'framework_validation_gpu_scale_artifacts.zip',
]

for path in expected:
    print(path.name, 'exists =', path.exists())

bundle_path = results_dir / 'framework_validation_gpu_scale_artifacts.zip'
print('\nbundled_download =', bundle_path)

report_path = results_dir / 'framework_validation_gpu_scale_report.md'
if report_path.exists():
    print('\n===== report preview =====\n')
    print(report_path.read_text(encoding='utf-8')[:4000])
